# Lista 13

## Multiprocessing, profilowanie

(6pkt + 2pkt)

Na liście znajduje się 6 zadań. Po rozwiązaniu zadań, pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** — tylko wtedy przyznajemy punkty. Dodatkowo prześlij zadanie na platformie skos.

Postaraj się, żeby Twoje wykresy były możliwie podobne do tych w rozwiązaniach.

**Zadania bonusowe**

Dodatkowe zadanie na wyższą ocenę oznaczone jest ⭐️ i warte 2 pkt.

## Identyfikacja wąskich gardeł czasowych (analiza)

Poniżej znajduje się fragment tabeli wygenerowanej przez profiler CPU:

```text
---------------------------------  ------------  ------------  ------------
                             Name      Self CPU     CPU total    # of Calls
---------------------------------  ------------  ------------  ------------
                     aten::conv2d     231.000us      31.931ms            20
                aten::batch_norm      211.000us      14.693ms            20
                       aten::mean     332.000us       2.631ms            21
---------------------------------
```

### Polecenia

1. Który operator jest największym wąskim gardłem czasowym?
2. Wyjaśnij różnicę pomiędzy **Self CPU time** a **CPU total time**.
3. Dlaczego `aten::conv2d` ma tak duży czas całkowity?

### Rozwiązanie

1. Waskie gardlo to conv23
2. Self to ile czasu spedzil w samej funkcji, a total to ile w calosci zajelo
3. To ciezka operacja ktora wymaga przejscia przez macierz 2d z kernelem

## Profilowanie zużycia pamięci (analiza)

Fragment wyników:

```text
---------------------------------
Name                     Self CPU Mem
---------------------------------
aten::empty              94.79 MB
aten::batch_norm          0 B
aten::conv2d              0 B
---------------------------------
```

### Polecenia

1. Dlaczego `aten::empty` zużywa najwięcej pamięci?
2. Dlaczego `aten::conv2d` ma `0 B` w kolumnie *Self CPU Mem*?
3. Czy oznacza to, że konwolucje nie zużywają pamięci?

### Rozwiązanie

1. wykorzystuje duzo pamieci, bo zwalnia pamiec
2. przeglada pamiec tylko
3. nie, to oznacza, ze nie pisza nic w pamieci.

## Uzupełnianie kodu: profilowanie CPU

Uzupełnij poniższy kod tak, aby:

* uruchomić profiler CPU
* zapisać kształty tensorów
* oznaczyć sekcję inferencji etykietą `"inference"`

In [8]:
from sympy import textplot
from torch.profiler import profile, ProfilerActivity, record_function
import torch
from torchvision import models

model = models.resnet18()
inputs = torch.randn(5, 3, 224, 224)

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("model_inference"):
        model(inputs)

print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=5
))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  model_inference         0.82%       7.485ms       100.00%     909.339ms     909.339ms             1  
                     aten::conv2d         0.03%     243.723us        63.18%     574.521ms      28.726ms            20  
                aten::convolution         0.31%       2.787ms        63.15%     574.277ms      28.714ms            20  
               aten::_convolution         0.06%     566.668us        62.85%     571.490ms      28.575ms            20  
         aten::mkldnn_convolution        62.69%     570.039ms        62.78%     570.924ms      28.546ms            20  
---------------------------------  -----

# Współdzielone tensory w multiprocessing

Uruchom poniższy kod i odpowiedz na pytania znajdujące się pod nim.

---

### Kod do uruchomienia

```python
import torch
import torch.multiprocessing as mp


def worker(rank, shared_tensor):
    for _ in range(1):
        shared_tensor[rank] += 1


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    tensor = torch.zeros(4)
    tensor.share_memory_()

    processes = []

    for rank in range(4):
        p = mp.Process(
            target=worker,
            args=(rank, tensor)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print("Final tensor:", tensor)
```

---

### Pytanie do analizy

**Sprawdź, co się stanie, gdy wszystkie procesy będą modyfikować to samo pole tensora zamiast różnych elementów.**

W szczególności:

* zmień kod tak, aby każdy proces zwiększał ten sam indeks tensora,
* zwiększ liczbę iteracji pętli `worker`,
* uruchom program kilkukrotnie,
* porównaj otrzymywane wyniki,
* spróbuj wyjaśnić obserwowane zachowanie.

In [23]:
import torch
import torch.multiprocessing as mp


def worker(rank, shared_tensor):
    for _ in range(1):
        shared_tensor[rank] += 1


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    tensor = torch.zeros(4)
    tensor.share_memory_()

    processes = []

    for rank in range(8):
        p = mp.Process(
            target=worker,
            args=(0, tensor)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print("Final tensor:", tensor)

Final tensor: tensor([0., 0., 0., 0.])


## Porównanie wykonania funkcji obliczeniowej – sekwencyjnie i z multiprocessing

Rozważamy funkcję obliczeniową `x(n)`, która wykonuje kosztowną operację numeryczną. Funkcja ta jest wywoływana wielokrotnie:

* w wersji **sekwencyjnej** (jeden proces),
* w wersji **równoległej** (wiele procesów, `Process + join`).

### Polecenia

1. Uzupełnij funkcję `run_sequential`.
2. Uzupełnij funkcję `run_multiprocessing`.
3. Upewnij się, że procesy są poprawnie synchronizowane przy użyciu `join()`.
4. Uruchom program dla:

   * `TASKS = 8`
   * `WORKERS = 8`
5. Odpowiedz na pytania:

   * Czy multiprocessing przyspieszył obliczenia?
   * Czy przyspieszenie jest liniowe?

---

## Kod do uzupełnienia

```python
import time
import torch
import torch.multiprocessing as mp


def x(n: int) -> float:
    """
    Costly numerical function.
    """
    t = torch.randn(n)
    for _ in range(5):
        t = t * t + 1.0
    return t.sum().item()


def run_sequential(tasks: int, n: int) -> float:
    """
    Run function x sequentially.
    Returns execution time.
    """
    start = time.time()

    for _ in range(____________):
        x(n)

    end = time.time()
    return end - start


def worker(n: int, out, idx: int):
    """
    Worker process.
    """
    out[idx] = x(n)


def run_multiprocessing(tasks: int, n: int, workers: int) -> float:
    """
    Run function x using multiprocessing.
    Returns execution time.
    """
    manager = mp.Manager()
    results = manager.list([None] * tasks)
    processes = []

    start = time.time()

    for i in range(____________):
        p = mp.Process(
            target=____________,
            args=(____________, ____________, ____________)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.____________()

    end = time.time()
    return end - start


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    TASKS = 8          # number of independent jobs
    N = 5_000_000      # cost of a single job
    WORKERS = 8        # number of processes

    t_seq = run_sequential(TASKS, N)
    t_mp = run_multiprocessing(TASKS, N, WORKERS)

    print("\n=== SUMMARY ===")
    print(f"Sequential:      {t_seq:.2f} s")
    print(f"Multiprocessing:{t_mp:.2f} s")
    print(f"Speedup:         {t_seq / t_mp:.2f}x")
```

In [24]:
import time
import torch
import torch.multiprocessing as mp


def x(n: int) -> float:
    """
    Costly numerical function.
    """
    t = torch.randn(n)
    for _ in range(5):
        t = t * t + 1.0
    return t.sum().item()


def run_sequential(tasks: int, n: int) -> float:
    """
    Run function x sequentially.
    Returns execution time.
    """
    start = time.time()

    for _ in range(tasks):
        x(n)

    end = time.time()
    return end - start


def worker(n: int, out, idx: int):
    """
    Worker process.
    """
    out[idx] = x(n)


def run_multiprocessing(tasks: int, n: int, workers: int) -> float:
    """
    Run function x using multiprocessing.
    Returns execution time.
    """
    manager = mp.Manager()
    results = manager.list([None] * tasks)
    processes = []

    start = time.time()

    for i in range(tasks):
        p = mp.Process(
            target=worker,
            args=(n, results, i)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    end = time.time()
    return end - start


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    TASKS = 8          # number of independent jobs
    N = 5_000_000      # cost of a single job
    WORKERS = 8        # number of processes

    t_seq = run_sequential(TASKS, N)
    t_mp = run_multiprocessing(TASKS, N, WORKERS)

    print("\n=== SUMMARY ===")
    print(f"Sequential:      {t_seq:.2f} s")
    print(f"Multiprocessing:{t_mp:.2f} s")
    print(f"Speedup:         {t_seq / t_mp:.2f}x")


=== SUMMARY ===
Sequential:      1.11 s
Multiprocessing:0.34 s
Speedup:         3.29x


## DDP + DataLoader z wieloma workerami

Twoim zadaniem jest:

1. Uruchomić **trening DDP** z wieloma procesami
2. Użyć **DataLoadera z `num_workers > 0`**
3. Zauważyć, że:

   * DDP **synchronizuje gradienty**
   * DataLoader **tylko ładuje dane**

---

### Kod do uzupełnienia

Uzupełnij miejsca oznaczone `__________`.

```python

```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler


# Dataset
class RandomDataset(Dataset):
    def __init__(self, size, length):
        self.data = torch.randn(length, size)
        self.targets = torch.randn(length, 1)

    def __getitem__(self, index):
        return self.data[index], self.targets[index]

    def __len__(self):
        return len(self.data)


# DDP setup
def setup(rank, world_size):
    dist.init_process_group(
        backend="__________",
        init_method="tcp://127.0.0.1:29500",
        rank=__________,
        world_size=__________
    )

# DDP cleanup
def cleanup():
    dist.destroy_process_group()


# Training function
def train(rank, world_size):
    print(f"Rank {rank} starting")

    setup(rank, world_size)

    # Dataset + DistributedSampler
    dataset = RandomDataset(size=10, length=1000)
    sampler = ________________________________

    # DataLoader with multiple workers
    dataloader = DataLoader(
        dataset,
        batch_size=32,
        sampler=sampler,
        num_workers=__________
    )

    # Model
    model = nn.Linear(10, 1)
    model = ________________________________
    model.train()

    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    for epoch in range(2):
        sampler.set_epoch(epoch)

        for x, y in dataloader:
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

        print(f"Rank {rank}, Epoch {epoch}, Loss {loss.item():.4f}")

    cleanup()


# Entry point
if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    WORLD_SIZE = 2

    mp.spawn(
        train,
        args=(WORLD_SIZE,),
        nprocs=__________,
        join=True
    )

## ⭐️ Optymalizacja kodu

1. Uruchom profiler
2. Odpowiedz:

   * która funkcja ma **największy `CPU total`**
   * która ma **największy `# of Calls`**
   * gdzie `Self CPU` ≪ `CPU total`
3. Znajdź wszystkie wąskie gardła i napisz ile znalazłeś/znalazłaś
4. Zoptymalizuj kod
5. Porównaj profil **przed i po**

---

## KOD DO PROFILOWANIA

```python
import torch
from torch.profiler import profile, record_function, ProfilerActivity


def level_3(x):
    # 4️⃣ alokacja + scalar sync
    y = torch.randn(1000)
    return (x + y).sum().item()


def level_2(x):
    total = 0.0
    for _ in range(50):  # 3️⃣ pętla
        total += level_3(x)
    return total


def level_1():
    result = 0.0
    for _ in range(100):  # 2️⃣ pętla
        x = torch.randn(1000)  # 1️⃣ alokacja w pętli
        result += level_2(x)
    return result


def run():
    with profile(
        activities=[ProfilerActivity.CPU],
        record_shapes=True,
        profile_memory=True
    ) as prof:
        with record_function("bzdurny_pipeline"):
            level_1()

    print(prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=20
    ))


if __name__ == "__main__":
    run()
```

Najwiekszy cpu total ma bzdurny_pipeline, ale funkcja to aten::randn

Najwieksza ilosc wywolan ma aten::randn, aten::normal_ i aten::empty - 5100

Nie wiem czy cos nie tak, ale kazda funkcja ma self cpu <= cpu total


Chyba, ze chodzi o funkcje level1, 2 i 3

Najwiekszy cpu total ma level1

Najwieksza ilosc wywolan ma level3 - 5000

level 2 i 3 maja self cpu <= total cpu


```text
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
             bzdurny_pipeline        26.77%      19.660ms       100.00%      73.438ms      73.438ms           0 B     -38.55 MB             1
                  aten::randn         9.81%       7.203ms        33.30%      24.452ms       4.795us      19.45 MB       1.98 MB          5100
                    aten::sum        19.88%      14.599ms        23.24%      17.070ms       3.414us      19.53 KB      19.53 KB          5000
                aten::normal_        19.79%      14.531ms        19.79%      14.531ms       2.849us           0 B           0 B          5100
                    aten::add        10.84%       7.960ms        10.84%       7.960ms       1.592us      19.07 MB      19.07 MB          5000
                   aten::item         4.41%       3.236ms         5.85%       4.296ms       0.859us         -40 B         -40 B          5000
                  aten::empty         3.70%       2.719ms         3.70%       2.719ms       0.533us      17.48 MB      17.48 MB          5100
             aten::as_strided         2.00%       1.466ms         2.00%       1.466ms       0.293us           0 B           0 B          5000
    aten::_local_scalar_dense         1.44%       1.060ms         1.44%       1.060ms       0.212us           0 B           0 B          5000
                  aten::fill_         1.37%       1.005ms         1.37%       1.005ms       0.201us           0 B           0 B          5000
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
Self CPU time total: 73.438ms

Po dodaniu record_function kolejnych

                         Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls
-----------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
             bzdurny_pipeline         0.02%      27.780us       100.00%     116.810ms     116.810ms           0 B      -3.91 KB             1
                       level1         0.40%     468.857us        99.98%     116.782ms     116.782ms       3.91 KB    -386.72 KB             1
                       level2        11.56%      13.505ms        98.83%     115.441ms       1.154ms           0 B     -19.07 MB           100
                       level3        38.46%      44.928ms        87.27%     101.936ms      20.387us      19.07 MB     -19.09 MB          5000
```

Najwieksze waskie gardlo to funkcja level3. Az 38% self cpu jest tam i 5000 wywolan

W samym kodzie jest tez duzo petli, ktore mozna zastapic wektorami. Problemem jest tez rekurencj, ktora pewnie tez daje jakis narzut.

Po poprawkach tak to wyglada. Jest o wiele szybciej.
Sam kod upraszcza sie do czegos takiego:

```python
def level_1():
    with record_function('level1'):
        return torch.randn(100, 50, 1000).sum() + (50 * torch.randn(100, 1000).sum())
```

```text
-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls
-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
       bzdurny_pipeline         0.15%      28.880us       100.00%      19.798ms      19.798ms           0 B          -4 B             1
                 level1         4.74%     938.765us        99.85%      19.770ms      19.770ms           4 B     -19.45 MB             1
            aten::randn         0.26%      50.820us        92.46%      18.307ms       9.153ms      19.45 MB           0 B             2
          aten::normal_        92.04%      18.223ms        92.04%      18.223ms       9.112ms           0 B           0 B             2
              aten::sum         1.99%     394.358us         2.24%     443.828us     221.914us           8 B         -56 B             2
              aten::mul         0.17%      33.770us         0.34%      66.510us      66.510us           4 B           0 B             1
            aten::empty         0.17%      34.520us         0.17%      34.520us       8.630us      19.46 MB      19.46 MB             4
               aten::to         0.05%      10.630us         0.17%      32.740us      32.740us           4 B           0 B             1
         aten::_to_copy         0.06%      11.430us         0.11%      22.110us      22.110us           4 B           0 B             1
            aten::copy_         0.11%      22.090us         0.11%      22.090us       7.363us           0 B           0 B             3
              aten::add         0.07%      13.940us         0.07%      13.940us      13.940us           4 B           4 B             1
       aten::as_strided         0.06%      11.340us         0.06%      11.340us       1.890us           0 B           0 B             6
           aten::select         0.05%       9.570us         0.05%      10.490us       5.245us           0 B           0 B             2
        aten::unsqueeze         0.03%       5.690us         0.03%       6.880us       3.440us           0 B           0 B             2
    aten::empty_strided         0.02%       4.710us         0.02%       4.710us       4.710us           4 B           4 B             1
            aten::fill_         0.02%       4.690us         0.02%       4.690us       2.345us           0 B           0 B             2
-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------
Self CPU time total: 19.798ms
```

Teraz bzdurny pipeline zajmuje 20ms, a wczesniej 73ms. Przyspieszylem kod o 3.65 raza